# Data Splitting — ABSA Hotel Santika (Google Colab)

Notebook ini membagi dataset hasil pelabelan menjadi **train / validation / test (80/10/10)**
untuk fine-tuning **IndoBERT** pada tugas *Aspect-Based Sentiment Analysis* (ABSA).

## Cara pakai di Colab
1. Jalankan sel **Setup** untuk install dependency.
2. Pilih salah satu cara upload dataset `dataset_absa_labeled.csv`:
   - **Opsi A — Mount Google Drive** (disarankan), atau
   - **Opsi B — Upload manual** lewat dialog file.
3. Jalankan sel berikutnya sampai selesai. Hasil split tersimpan di `/content/output`
   dan otomatis di-zip untuk diunduh.

### Justifikasi rasio 80/10/10 (dari literatur)
- Dataset menengah-besar (~13k sampel berlabel): 10% test (~1.3k) & 10% val (~1.3k) cukup
  untuk estimasi metrik stabil, 80% memaksimalkan data latih untuk model Transformer.
- **Devlin et al. (2019, BERT):** dev set untuk tuning, test set untuk evaluasi akhir.
- **Koto et al. (2020, IndoBERT/IndoLEM):** train/dev/test terpisah untuk NLP Bahasa Indonesia.
- **Pontiki et al. (2014/2016, SemEval ABSA):** pemisahan train–test standar.
- **Sechidis et al. (2011):** *iterative stratification* untuk data multi-label agar
  proporsi tiap label (termasuk kelas minoritas seperti **Harga**) terjaga di setiap split.

## 1. Setup — install dependency

In [ ]:
!pip -q install iterative-stratification scikit-learn
print('Dependency siap.')

## 2. Upload dataset `dataset_absa_labeled.csv`

Pilih **salah satu** opsi di bawah.

### Opsi A — Mount Google Drive (disarankan)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Sesuaikan path file di Drive Anda.
# Contoh jika file ada di MyDrive langsung:
LABELED = '/content/drive/MyDrive/dataset_absa_labeled.csv'

import os
print('File ditemukan:' , os.path.exists(LABELED), '->', LABELED)

### Opsi B — Upload manual (lewati jika sudah pakai Opsi A)

Jalankan sel ini lalu pilih file `dataset_absa_labeled.csv` dari komputer.

In [ ]:
# from google.colab import files
# uploaded = files.upload()
# LABELED = list(uploaded.keys())[0]
# print('Ter-upload:', LABELED)

## 3. Load dataset

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
    HAS_ITERSTRAT = True
except Exception:
    HAS_ITERSTRAT = False

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

OUT_DIR = '/content/output'
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(LABELED, encoding='utf-8-sig', dtype=str).fillna('')
print('iterative-stratification tersedia:', HAS_ITERSTRAT)
print('Total baris:', len(df))
df.head(3)

## 4. Filter review berlabel

Hanya review dengan **minimal 1 aspek terlabel** yang masuk train/val/test.
Review tanpa aspek disimpan terpisah (`no_aspect.csv`).

In [ ]:
ASPECTS = ['Lokasi', 'Kenyamanan', 'Pelayanan', 'Kebersihan', 'Harga', 'Makanan', 'Fasilitas']

def has_any_aspect(row):
    return any((row[a] or '').strip() != '' for a in ASPECTS)

mask = df.apply(has_any_aspect, axis=1)
df_labeled = df[mask].reset_index(drop=True)
df_noaspect = df[~mask].reset_index(drop=True)
print('Review berlabel (>=1 aspek):', len(df_labeled))
print('Review tanpa aspek          :', len(df_noaspect))

## 5. Matriks label multi-label untuk *stratified split*

Tiap pasangan `(aspek, sentimen)` menjadi satu kolom biner agar proporsi setiap
label terjaga di tiap split.

In [ ]:
SENTIMENTS = ['positif', 'negatif', 'netral']
Y = pd.DataFrame(index=df_labeled.index)
for a in ASPECTS:
    col = df_labeled[a].str.strip().str.lower()
    for s in SENTIMENTS:
        Y[f'{a}_{s}'] = (col == s).astype(int)
Ymat = Y.values
print('Bentuk matriks label:', Ymat.shape)
Y.sum().sort_values(ascending=False).head(10)

## 6. Split 80/10/10

1. Test = 10% dari data berlabel.
2. Validation = 1/9 dari sisanya (~10% total).
3. Train = sisanya (~80%).

In [ ]:
idx = np.arange(len(df_labeled))
if HAS_ITERSTRAT:
    msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.10, random_state=RANDOM_STATE)
    trainval_idx, test_idx = next(msss1.split(idx.reshape(-1, 1), Ymat))
    Y_tv = Ymat[trainval_idx]
    msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=1/9, random_state=RANDOM_STATE)
    tr_rel, val_rel = next(msss2.split(trainval_idx.reshape(-1, 1), Y_tv))
    train_idx = trainval_idx[tr_rel]; val_idx = trainval_idx[val_rel]
    method = 'MultilabelStratifiedShuffleSplit (iterative stratification)'
else:
    trainval_idx, test_idx = train_test_split(idx, test_size=0.10, random_state=RANDOM_STATE, shuffle=True)
    train_idx, val_idx = train_test_split(trainval_idx, test_size=1/9, random_state=RANDOM_STATE, shuffle=True)
    method = 'train_test_split acak (fallback)'

train_df = df_labeled.iloc[np.sort(train_idx)].reset_index(drop=True)
val_df   = df_labeled.iloc[np.sort(val_idx)].reset_index(drop=True)
test_df  = df_labeled.iloc[np.sort(test_idx)].reset_index(drop=True)

print('Metode :', method)
print('Train  :', len(train_df), f'({len(train_df)/len(df_labeled)*100:.1f}%)')
print('Val    :', len(val_df), f'({len(val_df)/len(df_labeled)*100:.1f}%)')
print('Test   :', len(test_df), f'({len(test_df)/len(df_labeled)*100:.1f}%)')
assert len(train_df)+len(val_df)+len(test_df) == len(df_labeled)

## 7. Verifikasi distribusi & cek kebocoran

In [ ]:
def dist_table(frame):
    rows = {}
    for a in ASPECTS:
        col = frame[a].str.strip().str.lower()
        rows[a] = {
            'positif': round((col == 'positif').mean()*100, 1),
            'negatif': round((col == 'negatif').mean()*100, 1),
            'netral':  round((col == 'netral').mean()*100, 1),
            'none':    round(((col == '') | (col == 'none')).mean()*100, 1),
        }
    return pd.DataFrame(rows).T

print('=== TRAIN (% per aspek) ==='); display(dist_table(train_df))
print('=== VAL   (% per aspek) ==='); display(dist_table(val_df))
print('=== TEST  (% per aspek) ==='); display(dist_table(test_df))

In [ ]:
s_tr=set(train_df['ID_Review']); s_va=set(val_df['ID_Review']); s_te=set(test_df['ID_Review'])
print('Overlap train-val :', len(s_tr & s_va))
print('Overlap train-test:', len(s_tr & s_te))
print('Overlap val-test  :', len(s_va & s_te))
print('Total unik gabungan:', len(s_tr | s_va | s_te))

## 8. Simpan & unduh hasil

File disimpan di `/content/output` dan di-zip agar mudah diunduh ke komputer/Drive.

In [ ]:
train_df.to_csv(f'{OUT_DIR}/train.csv', index=False, encoding='utf-8-sig')
val_df.to_csv(f'{OUT_DIR}/validation.csv', index=False, encoding='utf-8-sig')
test_df.to_csv(f'{OUT_DIR}/test.csv', index=False, encoding='utf-8-sig')
df_noaspect.to_csv(f'{OUT_DIR}/no_aspect.csv', index=False, encoding='utf-8-sig')

import shutil
zip_path = '/content/absa_split'
shutil.make_archive(zip_path, 'zip', OUT_DIR)
print('Tersimpan di', OUT_DIR, 'dan', zip_path + '.zip')
for f in ['train.csv','validation.csv','test.csv','no_aspect.csv']:
    print(' -', f)

### (Opsional) Download zip ke komputer

In [ ]:
from google.colab import files
files.download('/content/absa_split.zip')

### (Opsional) Simpan hasil ke Google Drive

In [ ]:
# import shutil, os
# DEST = '/content/drive/MyDrive/ABSA_split'  # pastikan Drive sudah di-mount
# os.makedirs(DEST, exist_ok=True)
# for f in ['train.csv','validation.csv','test.csv','no_aspect.csv']:
#     shutil.copy(f'{OUT_DIR}/{f}', f'{DEST}/{f}')
# print('Disalin ke', DEST)